# Imports Needed

In [1]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------- ----------------------- 5.2/12.8 MB 39.8 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 37.6 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 36.5 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt

# Loading Data

In [3]:
df = pd.read_csv("C:\\Users\\Carol\\AI-Hallucinations-Detection\\data\\cleaned_data.csv")

df.head()

,reference,input,output,label,hallucination_type_realized,question_type,hallucination_type_encouraged
0,"A.D.A.M., Inc. estÃ¡ acreditada por la URAC, t...","What organization has accredited A.D.A.M., Inc...","A.D.A.M., Inc. is accredited by the Atlantis H...",hallucinated,Entity-error hallucination,Default question type,Entity-error hallucination
1,"""This dataset for NOAA's Science On a Sphere d...",What type of educational activity is encourage...,The dataset encourages learners to complete a ...,hallucinated,Relation-error hallucination,Default question type,Relation-error hallucination
2,"Dimension items include ""Foreground"" and ""Back...",What is the primary reason for a hit to be cla...,The primary reason for a hit to be classified ...,hallucinated,Relation-error hallucination,Other common hallucinated questions,Other hallucination
3,"Atlas Search runs a new process, called mongot...",What are the specific hardware requirements fo...,"Based on our production monitoring data, mongo...",hallucinated,Unverifiable information hallucination,Other common hallucinated questions,Other hallucination
4,The business implications are stark. In a surv...,What percentage of banking executives in the l...,34% of banking executives in the loan originat...,hallucinated,Relation-error hallucination,Other common hallucinated questions,Other hallucination


# Named Entity and Numerical Fact Overlap Functions

In [4]:
#model we are using (reads people, organizations, locations, numbers, dates, etc.)
nlp = spacy.load("en_core_web_sm")

c:\Users\Carol\miniconda3\envs\spark-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


| Disabled | Reason | Example |
|----------|----------|----------|
| parser | It figures out the sentence structure, but we did not need this in our NER model | Subject, object, relationships between words |
| tagger | Labels words as noun, verb, adjective, etc. which is not needed for entity extraction. Good for grammar tasks though.| "Obama" -> PROPN (proper noun)|
| lemmatizer | Converts words to base form. Feels irrelevant for entity detection. | "running" -> run |
| tok2vec | Vector representation, don't need it | ---- |
| attribute_ruler | Ours mainly depends on tokenizer and statistical NER model | --- |

en_core_web_trf was taking quite awhile to run in our for loop when running the model with our data which we theorized that it was taking up too much GB in our RAM

In [5]:
def extract_entities(texts):
    """
    Extract entities for a list of texts using spaCy's nlp.pipe for efficiency.
    Returns a list of dictionaries whose values are sets.
    """
    ner_list = []
    for doc in nlp.pipe(texts, disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"], n_process=-1):
        entities = {}
        for ent in doc.ents:
            entities.setdefault(ent.label_, set()).add(ent.text.lower())
        ner_list.append(entities)
    return ner_list

In [6]:
def convert_entities(ents):
    """
    converts list of dictionaries into list of sets 
    who contain tuples of the form (TAG, entity)
    """
    ent_list = []
    for item in ents:
        item_ents = [(key, sub_val) for key, values in item.items() for sub_val in values]
        ent_list.append(set(item_ents))
    return ent_list

In [7]:
#testing how the function works
texts = ["Barack Obama was president.",
    "Apple is based in Cupertino."]

test = extract_entities(texts)

print(test)

[{'PERSON': {'barack obama'}}, {'ORG': {'apple'}, 'GPE': {'cupertino'}}]


In [8]:
def calculate_entity_recall(ref_entities, resp_entities):
    """Args: 
    ref_entities (set): set of tuples representing the reference entities in the reference text.
    resp_entities (set): set of tuples representing the response entities in the response text.
    Returns:
        float: The recall score for the given entities."""
    tp = len(ref_entities & resp_entities)
    if len(ref_entities) == 0:
        return 1.0 #if there are no entities in the reference, we consider recall to be perfect (1.0)

    return tp / len(ref_entities)

In [9]:
#out of reference function
def out_of_reference_rate(ref_entities, resp_entities):
    """Args:
    ref_entities (set): A set of tuples representing reference entities.
    resp_entities (set): A set of tuples representing response entities.
    Returns:
        float: The out-of-reference rate for the given entities."""

    if len(resp_entities) == 0:
        return 0.0

    return len(resp_entities - ref_entities) / len(resp_entities) #the response has that amount of entities that are not in the reference

Looking to see which spacy NER are numerical labels

In [10]:
unknown_explanation = ["CARDINAL", "FAC", "GPE", "NORP", "LAW", "NORP", "ORDINAL","PRODUCT", "WORK_OF_ART"]

for ent in unknown_explanation:
    print(f"{ent}: {spacy.explain(ent)}")

CARDINAL: Numerals that do not fall under another type
FAC: Buildings, airports, highways, bridges, etc.
GPE: Countries, cities, states
NORP: Nationalities or religious or political groups
LAW: Named documents made into laws.
NORP: Nationalities or religious or political groups
ORDINAL: "first", "second", etc.
PRODUCT: Objects, vehicles, foods, etc. (not services)
WORK_OF_ART: Titles of books, songs, etc.


In [11]:
NUMERIC_LABELS = {
    "CARDINAL",
    "ORDINAL",
    "QUANTITY",
    "MONEY",
    "PERCENT",
    "DATE",
    "TIME",}

def number_overlap(ref_entities, resp_entities):
    """Args:
    ref_entities (dict): A dictionary where keys are entity types and values are sets of entities extracted from the reference text.
    resp_entities (dict): A dictionary where keys are entity types and values are sets of entities extracted from the response text.
    Returns:
        float: The overlap rate for the given numerical entities."""
    ref_nums = set()
    resp_nums = set()

    for label in NUMERIC_LABELS:
        ref_nums.update((label, frozenset(ref_entities.get(label, set()))))
        resp_nums.update((label, frozenset(resp_entities.get(label, set()))))

    if len(ref_nums) == 0:
        return 1.0

    return len(ref_nums & resp_nums) / len(ref_nums)

# Applying model to our data

In [12]:
print(df.columns)

Index(['reference', 'input', 'output', 'label', 'hallucination_type_realized',
       'question_type', 'hallucination_type_encouraged'],
      dtype='object')


In [13]:
# Extract entities for all texts in batches
ref_entities = extract_entities(df["reference"])
resp_entities = extract_entities(df["output"])

# Flatten entity dictionaries to sets of unique entities
reference_ents_conv = convert_entities(ref_entities)
response_ents_conv = convert_entities(resp_entities)

In [14]:
ent_recall = []
oor_rate = []
num_overlap = []

for i in range(len(reference_ents_conv)):
    ent_recall.append(calculate_entity_recall(reference_ents_conv[i], response_ents_conv[i]))
    oor_rate.append(out_of_reference_rate(reference_ents_conv[i], response_ents_conv[i]))
    num_overlap.append(number_overlap(ref_entities[i], resp_entities[i]))

In [15]:
ner_scores = pd.DataFrame({"Entity Recall": ent_recall,
                           "Out-Of-Reference Rate": oor_rate,
                           "Number Overlap": num_overlap})
ner_scores.head(10)

,Entity Recall,Out-Of-Reference Rate,Number Overlap
0,0.000000,1.000000,1.000000
1,0.000000,1.000000,0.888889
2,0.500000,0.000000,1.000000
3,0.200000,0.900000,1.000000
4,0.333333,0.666667,0.888889
5,0.100000,0.833333,0.888889
6,0.000000,0.000000,1.000000
7,0.500000,0.714286,0.800000
8,0.000000,1.000000,1.000000
9,0.000000,1.000000,1.000000


In [16]:
#add ref column back for merging later
ner_scores['reference'] = df['reference'] 
ner_scores['label'] = df['label']

ner_scores.head()

,Entity Recall,Out-Of-Reference Rate,Number Overlap,reference,label
0,0.000000,1.000000,1.000000,"A.D.A.M., Inc. estÃ¡ acreditada por la URAC, t...",hallucinated
1,0.000000,1.000000,0.888889,"""This dataset for NOAA's Science On a Sphere d...",hallucinated
2,0.500000,0.000000,1.000000,"Dimension items include ""Foreground"" and ""Back...",hallucinated
3,0.200000,0.900000,1.000000,"Atlas Search runs a new process, called mongot...",hallucinated
4,0.333333,0.666667,0.888889,The business implications are stark. In a surv...,hallucinated


# Observations

| Feature | Low Value (close to 0) | High Value (close to 1) |
|----------|----------|----------|
| entity_recall | Response kept very few reference entities | Response kept most reference entities |
| out_of_reference_rate | Response introduced few new entities | Response introduced many new entities |
| number_overlap | Response changed or omitted most numbers | Response preserved most numbers |

In [17]:
mean_ner = ner_scores.groupby("label").mean(numeric_only=True)

print(mean_ner)

              Entity Recall  Out-Of-Reference Rate  Number Overlap
label                                                             
factual            0.340271               0.168025        0.934759
hallucinated       0.373306               0.478555        0.946144


In our dataset, hallucinated responses showcases higher average entity recall, out-of-reference entity rates, and higher numerical overlap than factual responses. This could mean that our hallucinated responses can still repeat many correct entities and numbers while at the same time adding false information.